# Top-k MoE router with load statistics and balance loss

A sparse mixture-of-experts layer sends each token to only $k$ of its $E$ expert MLPs. A small linear **router** scores the experts, and its choices decide both the output and how evenly work spreads across experts (and the GPUs that hold them).

Write `route(x, W_r, k, renormalize=True)` in PyTorch.

## Inputs
- `x`: float64 `(N, D)` token states (batch and sequence already flattened). `W_r`: float64 `(D, E)`. `k`: int, $1 \le k \le E$.

## Computation
1. Router probabilities $P = \mathrm{softmax}(x W_r)$ over experts, shape `(N, E)`.
2. For each token, pick the $k$ experts with the largest probability, **highest first**, ties to the smaller expert index.
3. Gates: the chosen probabilities. If `renormalize`, divide them by their sum so each token's gates sum to 1 (Mixtral-style); otherwise keep them as they are (they then sum to less than 1).
4. Load: `counts[e]` = number of (token, slot) assignments that went to expert $e$.
5. Balance loss (Switch Transformer form): $\;\mathcal{L} = E \sum_e f_e\, \bar P_e$, where $f_e = \texttt{counts}[e] / (N k)$ is the fraction of assignments to $e$ and $\bar P_e$ is the mean of $P_{:,e}$ over tokens.

## Output
A tuple `(idx, gates, counts, loss)`:
- `idx`: int64 `(N, k)`, `gates`: float64 `(N, k)`, `counts`: int64 `(E,)`, `loss`: Python float.

No Python loops over tokens.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def route(x, W_r, k, renormalize=True):
    """Return (expert idx (N,k) int64, gates (N,k), counts (E,) int64, balance loss float)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], "dtype": "float64"}, {"__nd__": [[2.0, 0.0, 1.0], [0.0, 2.0, 1.0]], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": [[0], [1], [0]], "dtype": "int64"}, {"__nd__": [[1.0], [1.0], [1.0]], "dtype": "float64"}, {"__nd__": [2, 1, 0], "dtype": "int64"}, 1.0886048622785356]}}, {"a": [{"__nd__": [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], "dtype": "float64"}, {"__nd__": [[2.0, 0.0, 1.0], [0.0, 2.0, 1.0]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[0, 2], [1, 2], [0, 1]], "dtype": "int64"}, {"__nd__": [[0.7310585786300048, 0.2689414213699951], [0.7310585786300048, 0.2689414213699951], [0.5, 0.5]], "dtype": "float64"}, {"__nd__": [2, 2, 2], "dtype": "int64"}, 1.0]}}, {"a": [{"__nd__": [[1.8267565599574231, -3.0783319101980338, 0.9580639753088469, 0.06963722766094482, 1.3182500241810684, 0.385629249998389, 1.8272586275861753, 0.0317437591517664], [-0.5162294444924808, 0.5804849213397179, 0.43210686133773885, -0.35683935740335093, -0.24730382198818454, 0.7194406781853278, 0.7043159938619936, -0.4939342302351804], [-0.3677137240199963, -1.8067903895865323, 1.6792074674884705, -0.2242908783928769, 1.337277430495411, 0.4174655938071864, 1.9439627746249157, 1.537109976128164], [0.318298439352904, 1.480763419858435, -0.9501235216099734, 1.2586181429890126, -1.4804236275921896, 0.3432363675742628, 1.064876469210243, 0.22363214164877185], [-0.3671374972389881, -0.8055600446794365, -0.3428000151424468, 1.051125142503177, 0.8908394462458891, -0.2621314629648405, -1.2460043473128628, 0.6739972418852872], [-1.4498739364757562, -0.530854904621622, -0.7348283771922636, 0.7432652102376149, 0.23594974138056524, 0.46185473227162593, 0.27240791046479607, -0.6779301162200867], [0.535475983047878, 1.4124603684044394, -0.03677124294459777, 0.6335944644542391, -0.125903192963739, 1.0285554273542712, 0.6666366521512817, 0.8758194707912133], [0.3484229955590589, 1.6400038821107275, -0.3611505786282483, -0.33416607721805125, -0.5918599835407998, 0.610962701968474, -0.6224002707066544, -0.6445326365001358], [0.7274748296203505, 1.1620957720506857, 0.5743514885621694, -2.683103555466855, -0.9536324496718003, -1.07348948063148, -1.2075111797268347, -0.4077676482371872], [1.8175432337026303, -0.3073985537659124, -0.7354457743627153, 0.6988227448389438, 0.06917172943950622, 0.12068112866070582, -0.6866741120790538, -0.0015044670379158913], [0.43453717981416906, -1.9113065771604514, -1.610224331010523, 1.0845605588518257, 0.8430045749240583, -0.5066567175591229, -1.020203904417219, -0.18554662570969443], [-1.1936522904412012, -0.3400969906062227, 1.651100423975727, 0.1320718435065325, 1.1662884363547947, -0.22262052870934315, 0.3694155619109898, 1.3858304929992982], [-1.3359375572222492, -0.34747799464982515, -0.1568985005454348, 0.28198117936536454, 1.553666722804394, 0.6855270035798681, -1.4976665985180717, -0.2709720411793551], [-0.8936434154830173, -0.919047029962896, 2.024827715003513, -1.0629706837475945, 0.23639517490996417, -0.7152122320787201, 0.8609809399519063, 0.35370323854134944], [0.1274277181041873, -0.07616714005920917, 1.1377169165755536, -0.9316998851391717, -0.7269383253215163, -1.1166060472205357, 1.334546412220603, -0.2935663794448887], [-1.03740077070428, -0.4190734003279602, 2.499647141912049, -0.10747295455808797, 0.8500205977374702, 0.4651818264795043, -0.012383282080929063, -0.6927049513484409]], "dtype": "float64"}, {"__nd__": [[-0.33429524042418257, -0.3544524150979252, -2.209276174952708, 2.590132165299634], [0.9248362579408633, 0.11628650676709631, -0.20334913839455843, -0.7896443817288501], [0.949621715476415, -0.18685598546529084, 0.47927255908121963, -1.3874951680999685], [0.3784765367574817, 2.000301160167004, -0.28962886468938337, 0.7999549486302511], [0.03533363711049965, 1.1051905403205458, 1.0674245093302333, 0.06164637112836233], [-0.559585430021553, 0.2867541614425335, 0.04176902945224784, -0.6179033056301174], [0.45647884748874545, 0.4323228618761901, -0.8890266047821327, -0.1114878979154479], [2.921139507657893, 0.8168344872544508, 0.09777931084435634, 2.511082373702184]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[3, 1], [2, 0], [0, 1], [3, 1], [1, 3], [2, 1], [0, 1], [2, 0], [0, 2], [3, 1], [3, 1], [0, 2], [2, 1], [2, 0], [0, 2], [2, 1]], "dtype": "int64"}, {"__nd__": [[0.986204232426648, 0.013795767573351944], [0.7256141431986592, 0.27438585680134076], [0.9089882664521047, 0.09101173354789521], [0.5646211657391296, 0.43537883426087043], [0.5185921214862234, 0.4814078785137766], [0.6621499721016465, 0.33785002789835344], [0.7697243077269561, 0.23027569227304384], [0.5752619156014233, 0.4247380843985767], [0.5016014656941711, 0.498398534305829], [0.9971684074997448, 0.0028315925002552014], [0.9695742440786057, 0.03042575592139433], [0.8296249238542123, 0.17037507614578776], [0.9778155269598552, 0.02218447304014485], [0.5317121983417893, 0.46828780165821066], [0.9208667810916182, 0.07913321890838182], [0.9871023606124123, 0.012897639387587794]], "dtype": "float64"}, {"__nd__": [8, 10, 9, 5], "dtype": "int64"}, 0.9696088417019086]}}, {"a": [{"__nd__": [[-0.45540319859640854, -2.47685411921746, -1.9069414995045355, -1.5195110137903052, 0.2600550542500467, 0.976356295342398], [-2.170190238668888, 1.1172437209364225, -1.2277384405743208, 0.25643484822861606, 0.463919602174066, 0.43232983369059774], [0.0075470802723938074, 0.17099551394253604, -1.139820149948546, 0.27821689039644487, 0.979652364759179, 0.9262226307647625], [2.6128529070721185, 1.0832325419401145, 0.7264273559501742, 1.6635005917178431, 0.24113763858859116, -1.2898772197929023], [-2.23985985733975, 1.2765709163818362, 0.5680313489899385, 0.847296954372218, -0.3328748962507801, 0.34482939616746583], [1.8823816030932006, 0.33483420185153473, -1.1155186762678087, -0.43144971563468354, 1.385110615586996, -1.089961064286938], [-2.58142946001913, -2.293955187945487, -1.3389389586186435, 0.7993935284932526, 0.08846320092734136, -0.7572771271592771], [-1.2851365294184347, -0.11398930813306221, -0.1595956330434021, -0.5639269440512877, -0.24525414605240478, -0.7631817772148304], [1.8032219405855618, -2.3669413655499887, 0.12218885596494325, -1.014405574517355, 0.7065164385024355, -0.23510619737365038], [0.36602144724812463, 0.531596243829064, 0.2053262383967149, 0.9910404617440158, -1.391477754463506, 0.9516989121641768]], "dtype": "float64"}, {"__nd__": [[-0.218942449625786, 0.38064437732387874, 1.5565333740979994, 1.124406837228961, 1.983906260402673, -2.250591833413329, -0.9558816529403455, 0.9641676674228251], [-1.130678125936227, -0.18259281381333614, 1.7078758612485556, 0.8746999899786561, -0.9622686699630217, 0.4538064407814815, 0.6453005002956944, 0.06737638601500909], [-0.10050634112829009, 0.1415574756124382, -1.5005136959043504, 0.12651241840846753, -0.9669110793525206, 0.39357107135464237, 0.42921695904368384, -0.892793268783839], [0.612671144108547, 0.3558124292565846, -0.04209628672662296, 0.7258067220827112, -0.31279669658828074, -0.28415798222731575, 0.07778617965586546, -0.059222803868094115], [1.142166871287435, -2.373945235389359, -0.3104735529634693, -2.0149561047575175, -0.5262386979768381, 0.2322282371772858, -0.022146193899912747, 0.9393318676256391], [0.2866276607088688, -2.7768745606713137, -1.1583429548412214, 1.5136624934334706, 1.5159204065480982, 0.05234707902313534, -1.601787701176551, 2.118804070102688]], "dtype": "float64"}, 2, false], "e": {"__tup__": [{"__nd__": [[4, 7], [5, 6], [7, 4], [2, 1], [5, 6], [2, 4], [5, 0], [5, 6], [4, 0], [3, 4]], "dtype": "int64"}, {"__nd__": [[0.7758959203677016, 0.1500909807089918], [0.9437712350531078, 0.032135846208853965], [0.7833071588256224, 0.0913715844563606], [0.8126657103505491, 0.1521658182267686], [0.9341780871294055, 0.05600025313582351], [0.9560175186388505, 0.021781212400279203], [0.5269281342068104, 0.3794720126251681], [0.48285733314356744, 0.2728730100342408], [0.9214776939078503, 0.04877433846112855], [0.9610371334501621, 0.01786614865202329]], "dtype": "float64"}, {"__nd__": [2, 1, 2, 1, 5, 4, 3, 2], "dtype": "int64"}, 1.21229891286702]}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[0, 1], [0, 1], [0, 1], [0, 1]], "dtype": "int64"}, {"__nd__": [[0.5, 0.5], [0.5, 0.5], [0.5, 0.5], [0.5, 0.5]], "dtype": "float64"}, {"__nd__": [4, 4, 0, 0], "dtype": "int64"}, 1.0]}}, {"a": [{"__nd__": [[-2.1530353066668972, 0.25024316589593604, -0.11187308559505371, 0.17806975537603983, -1.240717357266528, -0.4722427659931478, 1.0021493076522272, -0.566389865708578, -1.4614338387338917, -0.09685989180754231, 0.006268835734178283, -0.22062527245614327, -0.8773841377983873, 0.4550834628751101, -0.21320793667014518, -1.2648979159843472], [0.13264485194458006, -0.11691341642839587, 0.05893535902297446, -1.7931824180543972, -0.8577885375379697, 0.37299983596970493, 0.8514968812412, 0.26848824196572596, 0.08492853776235817, 0.6574750997519162, 0.9197622559268187, 1.7089095991597132, 0.416161651662968, -0.617898809061064, 0.43328855938185223, -1.994313337409666], [0.02022833988314914, -0.4773692452515349, -0.05159262496450286, 0.46636249718118594, 0.028592398183438686, -1.472240175171709, -0.3766682516719827, -0.8190875247325766, -0.035214233348543234, 0.34775832505667026, -0.9762750527390194, 0.7236144996200846, 1.2932844357111286, -0.2788844254279498, 0.7214635410570496, -1.4895076996078205], [-0.5475362446940503, 0.9577655289636221, -1.771726555102153, -0.5123825891381801, -0.9021143736434656, 1.1438186127113699, 1.1407461185846095, -0.6291699856677592, 0.11823820239791931, -2.6185953294711948, 0.8450719508280701, 0.34489307430605387, 1.321676514269612, -2.3936133348335895, 0.34245155509221265, -0.3584923321792362], [-2.617582772625446, -0.6984831500347612, -0.08622462961584804, 0.7974617928943865, 0.19020103547696696, -0.5223810280628554, 0.43909767776573405, -0.11440622715817586, -0.6565077024775354, -1.0265801957742764, 0.1428425333688465, 0.15089504441394663, 1.835772183308189, -0.45719439925974104, 1.599266519377065, 0.3460621048198438], [1.173685962859027, -0.9101956095226067, -0.8784043385444784, -0.3121608183309457, 0.8927077912890001, 0.17324927936697432, 0.750797171358702, 0.3159587283344176, 1.0272805308398631, 0.19700369537289608, -0.526489476375156, 0.5140479005699569, 0.09003370802947244, 1.7483366472512227, -0.15972213232813467, 0.10902445224740528], [-0.3064395127632179, 1.1561104824845763, 0.8746158488269457, 0.6510992265369949, -1.2466926303677017, -1.5165450265163778, 0.2648726884435707, 0.7220378712890259, 1.4791980565820146, -1.7956932334440707, -0.17030522858961525, -1.6904635961913856, -0.9607281602137127, -0.2211023076367421, 0.9630199134878099, -0.5266167805021937], [-0.718177505157109, -0.3198303871933791, 1.2576661536807427, 2.415912944780984, -1.5420102096218067, -0.27349778189704577, -0.43029893858656654, 1.1688016828523946, 0.4356662067659889, 1.5407292766608034, -1.0121275213254168, 0.013772513416587234, -0.45555510947930655, -1.0964166634549601, -0.15803399702278426, -1.1844993826725632], [2.1726217855985626, 0.417291116991025, -1.0164598086367858, -0.6060484283385521, 1.7326364891299604, -0.4781183586479403, 0.6941054669674107, 0.5836544246629394, 1.0927316672741476, 0.6347154356607344, -0.21011425768109862, -0.5914984158560455, -1.7757763891268423, -0.047652759873061314, 0.13870155113204108, -0.36019461661277574], [-0.1436373634512495, -1.2139527451556316, 2.171170181466329, -0.5010127696641761, -1.0353966715248002, -0.5970108050729186, -0.1684709087241838, 0.741899269403302, -2.427354701368989, 0.5157558345493929, -0.6158476669975436, -0.17923468221598868, 1.0174626752335119, 0.7666961115883858, -0.678179784303037, -0.0927329934736208], [1.8740146182883266, 1.3630983295296837, 0.8707621393904811, 0.011627346022443644, 0.9657787911057175, 0.400438161731701, -0.5452161686135586, -0.9970357958277322, 0.527059099674402, 0.38650282400685576, 1.2115569303421578, -0.4789942074159866, 1.4068457669695702, -0.36106491612390706, 1.26039495726865, 0.5507586823772175], [-0.5727424311955226, -0.9430860436137105, -0.11643341794836795, -0.2601644296113558, 0.9442658268166565, -0.6876183356133527, -1.47427343608644, -0.014948368277941878, 1.9265889310928719, 1.6422021791756498, -0.618040209628984, -0.2704604694931823, 0.32261649329743386, -0.6595926947340193, -0.24154739415968673, -0.2718432735704029], [-1.046504728135219, -0.6149859605204029, -0.48148909903628495, 0.23836323353673863, -1.1797867634185144, 0.10856445859123359, -0.9789382223230477, -0.8058544637440057, -1.121026295131995, 4.015010155012474, 1.2599880581787484, 1.34829093023627, 0.816188253132945, -0.23215779142042656, 0.0706855156052522, 0.29227174114881993], [0.4504415246356906, 0.5978002121627625, 1.5296722005868784, -2.9571879885470715, -0.552343563107178, 0.6801715744913809, -0.355494624846566, -0.1541263431168157, 1.4936681342175795, 0.19407388528120678, 1.7850669831230503, -0.04146758677129944, 0.23957011515021728, -1.0646258669403335, -1.985642268406066, -0.41670242449876116], [0.06352199839860827, -0.34391760405124766, -1.3467359093059932, 0.6585365188447696, 0.07284578203849948, 0.010964881384822552, -0.7770242299197428, 1.6502982933453774, 1.4791387728378758, 0.30928856620572487, -0.11463054321898763, 0.2541430273272719, 1.6978796790496515, 2.81183998514846, 0.013234156045813369, -0.4257902833664282], [-1.811583838559618, 2.7992971060959277, -1.103662097980581, 0.8742791830059452, 0.5016737785387297, 1.3359388864697388, -0.8596673959339524, -1.6201274916486155, 0.15047596535458788, 0.4127308508507303, 1.1513654794186778, -1.6848421647396326, -0.4526776608173918, -0.2618641347389855, -1.3526510274894763, -0.16133297789525175], [0.8022027087506243, -0.032154102422896354, -1.438142900884058, 0.3487955388990203, 0.8612946433858669, 0.5389732837048354, 0.7925052050347707, -1.6518425770283547, 0.46366350099286047, 0.6031314108936687, -0.3737937002002463, -0.0524281646582488, -0.7700425543731925, 1.2389658173489513, -0.29613097649865405, -0.942330386743925], [0.18934915688564294, 0.11445123938788547, 0.6718542517194702, -0.8322688549154442, -0.6745135578940894, 0.38671152263193265, 2.240921208979784, 0.47039511412656004, -1.3610966016695314, 0.17207482256510426, -0.7765471823216978, -1.8503470453903896, -0.6822057034052179, -0.5729925595396194, 0.4217438177703415, 1.325167275754661], [1.1429765955888342, 0.8132300211119146, -0.42983051139722905, 0.15966149141432187, -0.036658348893580335, 1.4138050502351096, -0.6498426031146902, -0.5474255936126069, 1.5014197442153803, 0.5814625698379817, -0.44623311727361414, 0.16897634142565574, 0.1537400391349979, 0.19328795017164876, 0.8058782701644918, 0.37177366273142204], [-0.17953971935450827, -0.5024283614966899, -1.7252230424895392, -0.4327968135284385, 1.151125889295179, 0.23226463292127603, -0.6488568874669097, -0.2384731672399161, -0.8530845952807655, 0.2961381258261375, 2.0159146263729197, 1.816286834932198, 1.2076217555801745, 0.7899340034941055, -0.21979404075334577, -1.2355076838721122], [0.019639477398621336, -0.4406792082535112, 0.842354235946767, 0.16933011186697836, -0.5659253533757543, 0.4640160572220805, 2.9363457620566757, -0.8966197143427083, 1.150119051570713, 0.6092477438757968, -0.08420709821669081, -0.7451052123552219, 0.03494151110915539, 0.23494111128265266, 0.7859031023939663, 0.7207475812968511], [0.33564357890679575, -0.6681472932385154, 0.3184939116611428, -0.4522288897623889, -1.0305270619785045, 0.9123314457807333, -0.6508362752464484, -1.1833113876743806, 0.09823982411163722, -1.9197437838688496, -0.05340863615666007, 0.4602773951383935, -0.40993490846590813, 0.24871130295322227, 2.1672805591578745, 1.4355412372838243], [0.9425172327212941, 0.6092635217456658, -0.5095141940345294, -0.13221036443500894, -0.27942388619165087, 0.033601392184566134, 0.12272394196017766, -0.8049627715859253, 1.6188916199509293, -1.0382230230687854, 0.3666912002302764, 1.037862493079551, -0.02648360207426057, -0.6680335013334684, 0.03341942377308756, 1.9428584793652173], [0.6261038742901202, -0.28960691934390487, 0.5850505747692876, -0.19636918699200892, -1.876551104266725, 0.24435110320669473, 1.0430858597689079, 1.1554746665574402, -0.06525294918659616, -1.4129848556083906, 1.1992993533250758, -0.1549652383551284, 0.7376782249413567, 1.1309545685582618, 0.33860929936217016, -0.06246346935367249], [-0.6303871788659644, -0.4355660213501193, 0.028307016526326802, -0.46638649211233957, 0.36001847749409543, -1.9614372833172928, -1.0189257977599415, -0.03862045469875908, -0.7034804255305075, -0.5474863513851672, -1.1492044955566831, 0.6610785358283214, 1.6204194916419912, -1.0199918485274655, 1.136606586106104, -1.4700734395858903], [0.12453320908131835, -2.0203676135941415, 0.4688175003970324, 0.0036097731144913866, 0.12210457678890772, 1.198567568003026, 0.5728311808598204, -0.04823875363942963, -0.8424930898996278, 0.02666337478187983, -0.7181218767193329, -0.44833970485403773, 1.4526640932855484, 0.8495865882822946, 0.2602483334529608, 0.6003328430131956], [-1.4725618512343468, 0.07622444926587726, -1.7234350498386277, -0.6984401894518526, 0.33611527201632196, 0.34210527905219806, 0.6642372197643215, 0.11605605835600245, 0.008802109521588164, -0.04673934865896081, 1.3597087543886883, 1.2530687289759483, -1.5053164054615977, 0.07836553902191477, 1.2380720507146512, -0.3105748303546403], [0.1627159257650581, -0.6616724441055528, -1.2639461684142952, 0.06622907317773313, -0.48092004167357966, -0.23853801156874002, -1.5294903642549813, -1.2471624560051209, -0.22655127538530925, -0.5447095358868417, 0.44337559418046374, -1.1642268657734398, 0.5917767468076053, 0.4475606360954432, -0.033190790226199036, 0.5150267552359696], [-0.21642622499878592, -0.33217651441293256, -0.16966303854692988, -1.4113640977588804, -0.1880147965164542, 0.6438881459373458, -1.7600427988479947, 0.5280057933274475, -0.7939373727122362, -0.5173986485588965, -0.07322857941875446, 0.08427481267345066, 1.197696272326208, -0.3094263811474142, 1.5460668680322236, -0.12257531507154988], [0.3228130534453869, 0.5140435015887131, -0.822980133975453, -1.271087891801766, -0.03993978314912153, 0.5501641860624883, 0.7179550479476372, -0.49595627607658777, 1.6254195024635476, -0.24865861278576082, -0.7929532506026082, 1.053194951517796, 1.1273050564395466, -1.0327833801392574, 0.7539494770986807, -1.3265343513204304], [-0.011146479022979544, 2.222589115326952, 1.2562011543243692, 0.6820922951215462, 0.6328900276714821, -0.830162463001373, 0.3328773758313582, 0.45354011915123343, -1.022486107109176, 0.21752508632188894, 0.5647282773951592, 0.2361591663450234, 0.45393899981152125, -0.19956548914265024, -1.0907214686420645, 0.06515006679147742], [-1.8843029037210206, -0.6782270445972737, 1.128256762963162, 1.8225554571078957, 0.4477506552094504, -2.363874246968447, 1.3792015586733108, 0.5875311042475977, -0.14244889630918978, 0.7963337295502729, 0.5948165845200453, -1.140617173982549, -1.2444142783062968, 1.0896666795356893, -2.2374739964263606, 0.6528853322731564]], "dtype": "float64"}, {"__nd__": [[1.1121567901952958, 3.104972702466001, 5.396692461991338, 4.186226648933943, 4.846549092118605, 2.6547607609907677, 1.7180955588434044, 3.4166871913929624], [0.18292313015539508, -2.512631963914558, -0.01541473555559294, -4.5701784195249004, -1.4288302558890291, 6.401307975233475, 1.7696591462120606, -0.31203788795453713], [-1.5858209012295115, -5.0963836078310765, -1.4071656228471736, -2.127939661096984, 0.18875310511610585, 0.42493281083505385, -0.024211193021933277, -5.548262742275002], [4.518210310840457, 1.4857242330267002, 0.18331801310504015, -0.8178641855288757, 1.4671775237390328, 3.8756108408369516, 1.9575622726062405, -3.343008206461298], [-3.630631515537494, -6.960078572305162, 0.47892801277025476, 2.2299663112791976, 3.726270207985188, 0.46541830327567035, 2.501436177440284, -2.69778921778412], [1.5969486859543203, -0.34443775823584677, -1.9745399016582357, -5.021209254797015, 3.4695948853091734, -6.749596001589412, 1.8758987150283335, -2.406657033023195], [1.8174681572047735, -1.6649499399294019, 2.2504861338041025, -0.660416319795112, -3.1431691890555085, -1.5620747872985825, -4.84644397616646, 0.8094798281934339], [3.1083111018342247, 1.2428336876222703, 0.30465636076002767, 3.8524056068863857, -3.5114476268093266, 1.6521847259805118, 0.015338621061714866, -1.424715244151601], [1.6355889615727879, -3.4455051710205833, -0.41402301987616213, 1.0009962820559728, -3.505258677164787, -0.7139000427801245, 1.5677567001961328, 2.8452227468501974], [-2.8369626919160935, 2.687854385766448, -0.8946825991327778, 0.26915949221192964, -1.6555300479233743, 2.331500574951734, -2.1248261062595577, 0.8383263165280912], [1.1266473454540171, 2.254942402953014, -3.128507963412811, 2.3108438081544427, 0.08849152988758506, 4.343819648317745, -1.0430419357638316, -1.7161549578046111], [-0.1841779302060158, -0.951367061975881, -0.5239835111623721, -3.8213809589846703, -6.262765072413835, -4.424345824910368, -1.7943242177142855, 1.2634612645600902], [-0.9589639045076559, -2.9041761358504505, -0.4928537360041474, 0.16037985336230462, 8.257425117820269, 3.810192785197022, 0.5372745906650176, -4.389216330518904], [-0.5533241672415347, -0.9397915903740564, 1.6396885173826, 6.341003661304514, -0.22129655268812773, -5.936111430057583, -0.37426577110891546, -5.055259141540143], [2.4922027075950415, 3.293591808882689, 0.9862106069223362, -4.473978741266764, -0.5500441169248831, 3.62648512493028, -3.0226873609646363, -1.787458115955633], [0.7565774384660295, 1.4822980443780243, 0.23032860155772827, 1.0130490727306318, 0.6183363519901709, -1.7465320546683918, -0.25055651047460353, 0.8484180058437321]], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": [[1], [7], [5], [0], [5], [3], [5], [5], [7], [4], [5], [3], [1], [7], [3], [6], [2], [1], [6], [3], [0], [1], [7], [0], [5], [4], [1], [4], [4], [7], [5], [3]], "dtype": "int64"}, {"__nd__": [[1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0]], "dtype": "float64"}, {"__nd__": [3, 5, 1, 5, 4, 7, 2, 5], "dtype": "int64"}, 1.1856423587135358]}}, {"a": [{"__nd__": [[-1.0921801327480034, 0.19398985538298488, 0.09719631067820901, 0.46132694084477743], [0.3477560405742926, -1.0491351053417286, -1.2584390760312487, 0.2500827203251652], [-0.15061555116968708, 0.10054108303346919, -0.5635381123808635, -1.2778362041704403], [0.2675737809190138, -0.13210625092462527, 0.6137542092486605, 1.0908142970265367], [1.091813948315874, 0.9876492263025507, -0.9601809821899305, -2.1120849076728043]], "dtype": "float64"}, {"__nd__": [[-0.6170183349512459, -1.2131604113993344, -1.3232161644747296, 0.6497148295723288, -1.6165414834683431, 0.9054979694096403], [1.087734557634296, 2.653220771803469, 1.0304797126558276, 2.088303496387598, -0.6487269683244603, -0.6892934468337224], [0.9701081174413879, -1.3215716271672076, 1.1325098418021207, -0.28140175028914755, -1.971976240288748, 1.8421061235774028], [-1.1265901263004665, 2.4894070969236695, -1.4824445377746884, -0.8530563798036775, 0.20653612436076926, 0.006309758603186645]], "dtype": "float64"}, 6], "e": {"__tup__": [{"__nd__": [[1, 4, 2, 0, 3, 5], [4, 1, 5, 3, 0, 2], [2, 3, 0, 4, 5, 1], [5, 1, 0, 3, 4, 2], [3, 0, 2, 4, 5, 1]], "dtype": "int64"}, {"__nd__": [[0.634461503850749, 0.17010743890330657, 0.10610058062770845, 0.057545705858182694, 0.01759694467657333, 0.014187826083479984], [0.9383635039550658, 0.02638084877420471, 0.01842831122611162, 0.010678754503404941, 0.0037966500471568985, 0.002351931494056084], [0.32003626182946, 0.26247952062633556, 0.20126497172243674, 0.1877366142006685, 0.019251303171805994, 0.009231328449293339], [0.48535205613340754, 0.38118605998810057, 0.043456338732327106, 0.03340731773539369, 0.02943500099507137, 0.027163226415699772], [0.9131703239256574, 0.04568293730187637, 0.03622683728755171, 0.0027861877995616472, 0.0016468961877395961, 0.00048681749761325194]], "dtype": "float64"}, {"__nd__": [5, 5, 5, 5, 5, 5], "dtype": "int64"}, 1.0]}}, {"a": [{"__nd__": [[-1.5863793166762856, 0.10066196063730329, 0.17944872992558708, 0.447569081510612, -1.6133707261226176, -0.6118007963116816, -0.11296556981287287, 0.7982983441244017], [0.15912777901422803, -0.35303780034481536, -0.43053119468679923, 1.4529585499742446, -0.8240624516156021, 0.2922510361945122, -2.174252207762989, 1.6942808979745667], [0.4529268876602559, -1.7648209886297261, 0.5994301968655034, 1.3820371855309714, 0.8122643747596423, 0.07000603610254197, 0.8404345946568171, -0.7480292125941229], [-0.2588430606595607, -0.36221196956310797, 0.07093449334091549, -1.0481683370244268, 1.372219458282659, -0.12945406229924541, -1.0834840208627612, -0.22282361839145826], [0.3603581266680253, 0.04042252487218727, 1.0949476284976696, -0.36199730403698577, -0.7752352874136524, 0.49611728650456277, 1.2738149428417191, -0.971097030996387], [-1.5918552043340808, -0.6492876440101472, -0.08174177005632867, -1.0354732541052547, 0.15550250368792715, -0.8705101933999353, 0.8839381305298087, -1.195253552832811], [-0.9227498640919723, 0.0612539424578776, 0.6640990442610537, 0.9097403334143831, -0.7009552644710758, 0.0058866303826475, 0.5257308454993269, 1.4118064779618447], [-0.9054897881245736, -1.260889614626335, 0.467703789658135, -0.2671178870048047, 1.3314962925464022, 0.00874015142384245, 0.3080451122371888, -0.8016861022383626], [0.2625436220444057, 1.2063271082827145, -0.2647437255583344, -1.209377870357122, 0.8669277548109309, -0.3272186423927756, 0.03759843255363001, -0.7326170829774914], [0.9229820669974117, -0.7643223011712551, -0.5017606346357297, -0.01240064352365647, -0.22153168946737758, -2.2626779930584546, 0.4496886796366645, -1.1867919801183435], [0.2365968887575378, -0.4110903311465741, -0.14485999724961965, -0.7568107750764189, -1.0839677450803589, 0.12151785196605354, 0.19358517780193907, 0.9865535553038376], [0.5155532123478671, -0.7235080945395366, 1.1442937222846292, 2.1819169939734877, 0.6932531996955813, -0.21459431668331794, -0.991255200078248, 0.0638539459708423], [-1.4611168332019289, 0.7682124996215773, -0.40619339546306604, -0.6010714124058425, 0.23076601252341297, -1.267837382728297, -1.7672612127072413, -0.46547597371254695], [1.187837165147437, -0.8773795066551493, 0.8411125190548874, -1.8671338806040076, -0.0945546822964162, 0.48021115189860675, 1.7549730629206726, -1.7903146891804573], [-0.09582647179384489, -0.9374459720703414, 0.2737713939678245, -0.5836781401294706, 0.07032899086817905, -0.7871711912142504, -0.5858621993216913, 0.367042325581881], [1.4570567575132825, 0.17598599990898917, 0.08886750621952007, -0.49562784454898484, 1.5427303339119824, -1.9707074720225903, -2.021507933679854, -0.8112369472286685], [0.4069746527490191, 0.9122158381320012, 0.5340163422988192, 0.7175655530450319, 0.13044177883940608, -0.5012674110646719, -1.0909396249643848, 0.4139263227812936], [-0.9783605607265474, 0.3858041071117823, 0.706590534371593, -0.648687476733967, -0.47927506843371487, -0.5811485014987571, 0.648706244110252, 0.12247626333436867], [-0.698878639376126, 0.23322931842348696, 1.4685178777641534, 0.24756722895058073, 0.09258831289446436, -0.43093944738370776, -0.9590957911753838, -0.028836945364434638], [0.03355853155749635, -0.33990268266173645, 1.1443601833926202, -0.26460505071375723, -0.1906483089006329, 0.16042468464285545, 0.4972982974016766, -1.8128712956395931], [0.23078995896493823, 1.2150661146617416, -1.1373785011079212, 0.35712364264095703, -0.5346667386650368, -1.013602276891089, 0.23339647703108302, -0.12935311695373836], [-2.0938385487677893, -0.6124882025576837, 0.7796835440916385, -0.6108347787593768, -0.7666376871408308, -0.39405336512606165, -0.4681606020282494, 0.7104989613878224], [-0.8697062111525586, -1.1843338719303562, 0.4018621550726627, 0.9591366512070939, 0.6692366273829996, 0.8073504913250081, -0.6679377371273234, -1.483552455593329], [-0.7196649056586572, -0.045222999146301616, -1.7043128686500855, -0.3210396425953245, 0.5551076895783724, -0.7054114675060696, 1.9473064375482052, 0.7237274457387506], [-1.2243596091554398, -1.1400083917095507, -0.7189697882770643, 0.6822138029508517, 0.4425178101703068, -0.5736066197908605, 0.14966907083730735, 0.7729075459017194], [0.49959390752182486, -1.60731232621174, -1.0937540230083027, -0.9238395818785466, -1.7291212510098914, 0.831201431347162, -0.6552144372851285, -0.6666637027123806], [-0.8465259383546485, 0.28037118407347944, 0.5423267214828222, 0.6492924910896203, 1.2066367163851641, -0.9398483477792341, 1.5910676822299485, 2.4362137338647236], [-0.5414386821465581, -0.37358731290687913, 0.5808382756745316, 1.2108524932152216, -0.2818839439585845, -0.8142946252748537, 2.0458638709871546, -0.5110553291667891], [-2.7332220814370967, 0.518319247177244, 0.9765257755435613, 1.6838244211659759, 1.7480032831907348, 3.0612001611561706, -0.5626429925258153, -0.9858656479294768], [-1.454087860573477, 1.692838600069469, -0.44145422957594393, -1.6794967019727571, -2.057642372584601, 0.11891292672774637, -0.43618393931829996, 0.3052470150403078], [-0.862024196099427, -0.741586328439177, 0.45976557581487726, -0.2084062783596275, 0.5679107525055531, -0.13550606202361626, -0.2929134989806901, 0.3522234397991429], [-0.33385764959604597, 0.16878901202783902, 0.43750356941591434, -0.5175598189640513, -0.1254808017163256, 0.09393626721952818, 0.6600291391723005, 0.22660307130794458], [0.2360398335865409, 0.4526379254511009, -0.3192278810307678, 0.34548914341927406, -0.742755054395803, -1.8258567082044714, 0.3977758666320973, -1.9598420975538728], [-0.1646690377145798, 1.5149200346460965, 0.707718681856061, -1.024963118023412, 0.7204521176889646, 0.22499940588792916, -0.8434364201086116, -0.10987175516283922], [-0.5424540718763711, -0.32217487735798855, 0.31953383132111934, -1.5536658080747185, 0.8300284720414973, 1.9890038676887078, -0.16686052386299233, -0.9768466860041071], [-0.41412156994474897, 1.210583569467275, -0.5079039995634741, 0.46677053401019997, 2.26378921517635, -0.723172409396254, 0.09705383498685011, -0.5700692091123941], [-0.36967752318795427, 0.11708650800853032, -0.7418762425977082, 0.4523126211161353, 0.17620611051209947, 1.1612439229959046, 0.6839698409411366, 0.383012545941538], [1.4962236924751464, -0.18894370480075595, 0.6344724692879731, -0.6307652410237127, 1.4336544630735206, -0.7074540359800024, 0.9859297904386043, -0.6929429464784617], [0.24380529645080073, -1.0253358165568947, -0.7271009939408893, -0.9044414791849353, 0.4768653649092368, 0.5680517823863174, 0.7994596290579016, 0.4490066640639636], [-0.1483257182561464, 0.7238884910093198, 1.6108271015765594, -0.30399174031254855, -0.6673916765686747, 0.6379453594414263, 2.1105911220730658, 0.002404993620663014], [-0.23644722347962932, -0.9114506494401993, -0.6265468684197165, -0.8414253043069891, -0.8643728996568122, 1.1450847663594477, 1.3597194130710621, 0.20086640751901635], [0.2067112931367893, 0.9218200609599163, 0.13083303463311663, 0.22337611287216216, 0.7104780020819267, 0.07873336938196414, 1.3994570573165495, 3.343346969278473], [0.12473997838597153, 0.4054220965574924, -0.6959803648431427, -0.31734447219844664, 0.2171364274266157, 0.547933847070797, -0.056604343708782694, -1.0545188991200112], [0.5163817953798187, -0.3962906943175209, 0.5369794677479104, -0.08610078970872637, 0.005029276239052044, 0.05587447637452819, -1.9752675971584222, 1.3633258656723257], [-0.7710289387227626, 0.49171384707386667, 0.6834228805564667, -0.4276729300196641, -1.2165102274705073, 0.7069383089458462, -0.8167572445873194, 0.05048482584798199], [0.5852422835296486, -0.3454992285247661, -1.5791666202887351, -0.5497377377543355, 0.6179208296549185, -0.687565732861304, 0.9896419915219057, -1.4107715377216408], [-0.8463734202869028, -1.637483140168055, -0.8460994501345604, 1.603207761713352, 0.6579159987712508, -0.18076545138035205, -0.5411076833180773, 1.217711349361244], [-0.04887506229214347, -0.9169337892297977, 1.608308548440255, -1.2434448636330482, 0.25046828280763817, 0.700021958039732, 1.0181290460764405, 2.01881426244981], [0.8900254481320802, -0.6391757854077854, 1.195995590848502, 0.35024526519422383, -1.1676905162666995, -1.0283947274699015, -0.7655250832924352, -2.111659104404517], [0.4663705172818137, 0.5945751475763261, 0.5197608903449298, -0.7922329013186556, -0.023962405043461494, 1.215963848373626, 0.404192894259565, -0.11806652301987336], [-0.7136350942572564, -0.01858797517612932, 1.2941796964611243, 0.04682822503344448, 2.3065912399091917, 2.4369754263926637, 0.7345381947983506, -1.6007831187807962], [-0.23558059796117498, 1.2917668276861505, -0.5365603794542061, -0.6670300046303318, 0.04185165942220386, 2.6933676800021056, -0.4537693084640108, -1.925538854806925], [0.30289274439259883, 0.41309436377821074, 0.7574178990441466, 0.08920022697784143, 0.35836977070237, -0.45337392716768316, 0.9728851390261907, -0.012046006509904447], [0.5285264431312893, -0.725041097654383, 1.4016215399951752, 0.2560257636590528, -0.4175787842084789, 0.05601584285628447, -1.0895644464198917, -0.591646219012702], [0.8337103402903148, 0.6148226621797465, 0.8461495696593955, 0.7177664195913995, -0.26892183588153773, 0.11134298277260923, 1.8966385065459506, -0.10079926804076794], [-1.527734447473783, -0.011885493984736446, -0.41734457242730716, -0.10999349052880909, 0.009748228662603435, -1.0405704319684583, 0.4064355589483934, -0.183336787875997], [0.5692316050708658, -0.719618997804702, -1.0147780739794647, -0.4598208886522349, 1.6190237506750682, -1.8392267477588884, 1.0560879647374903, -0.09007644971687633], [1.1082093921661733, -0.4349433859196837, 0.341260735630794, 0.3171912871450938, 2.072965846400098, -0.1733754896001823, 0.18565768916751815, -0.8161389213206458], [0.24527407179579958, -0.797849093907738, -0.1701124047980227, 0.3251000060246638, 0.5010623247768816, -0.9618199617732827, -0.9415929361304821, 1.108639103182709], [-0.581453131870384, 0.12439379373514889, -0.3385330200506401, -1.6272461156277076, -0.16101756387990085, -0.17671400688176836, 1.7098890019198958, -1.9512142352681945], [-0.5820895741168736, 0.5601062354076899, -0.5887768695671071, -2.2030771369171074, 0.5634773897442055, 0.11301595360295814, -0.9000091788987304, -1.04780376877346], [-0.7392123016991872, 0.13294117597780786, -0.23677732089883674, -0.716321777814637, 0.2789651813859545, 0.9772326267417555, -0.6632502183121483, 1.4899704234120434], [-0.03598016381221036, 0.5281570877892227, 0.34779206927002787, 0.4249107763672284, 0.16110846268579107, -2.32207392523734, -1.0212075235112954, 0.7946293714715298], [-1.1193976840851414, 0.027370559439419236, -1.6342350719041543, -0.6981988267696487, 1.4370665992026466, 0.15244001609793786, -0.9525130627496412, -0.7268542047882686]], "dtype": "float64"}, {"__nd__": [[-1.7801212155528066, 0.35588321310712545, -0.3465251813217331, 0.6166972254884181, 0.6755572757464419, -0.40976437151714934, -1.0668455776446533, -1.3211507042137407, -0.5260226009589481, -0.11231470457726339, 1.2370514279936793, -1.0799612438974868, 0.3137982784284902, -1.174508584189374, -0.7127521972093199, 0.6289868115410946], [0.37208609195230535, -0.6709147674737234, -0.7879875421874037, 0.9521892672747223, -0.3231487309542855, -0.4494050516225535, -1.1935860870363162, 2.3457138698681193, 1.6887389584290549, 0.38249927398108835, 2.082157966268343, 0.366469267803259, -0.45379782977831035, 0.4309796213680496, 1.3081956971014415, -1.2055360613172417], [1.1421812054196108, 1.0984951403872385, 1.4172620784387564, 0.4132705345867553, 1.0435432957437385, -1.779272726196454, -0.1130832785262235, 0.9729127653995959, 0.5013329234442544, 0.22494368030564973, -1.2804738137791871, -0.20711612589105852, -1.7590967937198643, -0.2778321838708371, 0.2951955052164264, -0.5500257678655625], [0.44566418421483345, -0.02577967379922399, 0.8578847434857834, -0.37911582838842295, -1.6262537431249375, -1.2840887570606074, -0.7869983075733795, -0.026459596478722168, 0.07307395431449554, -0.7598562740728282, 0.09142988916658032, -0.6729117411879588, 0.6250157089375511, -1.339340608847401, 1.14995579322779, 0.2749406995898913], [-0.3953324842085779, -1.0692309356073129, 0.5591391342238443, -0.5808485852689883, 0.665621609232966, 1.8508686639342982, 0.20305289216955874, 0.2776289193059825, 0.5845356287215049, -0.47953420916629097, 1.9907095665912606, -0.14436846606526782, 0.7674277652384532, 0.4234981463482353, -0.48408953219480455, 1.4619697935574363], [0.48624747506325405, -0.5275807890653692, -0.5163500268035012, -1.7783744348721873, -0.9224791903310299, 0.9791021849752357, 0.9944040371793242, -0.7681300540281453, 0.7822249857253155, 1.136113142980695, 0.9392929594149595, 1.4147726069212896, -1.7190144223148882, 0.37395150689290035, -0.7750612347821505, 0.8448583961552157], [-0.5652911929829615, 0.27189154429322, 1.040712298005342, -1.398753294089671, -0.4857615328876701, -0.5406025852718067, 0.115444766307235, -1.4176619416370684, 0.9189150067995453, 2.048274615139033, -0.15380518104897845, 0.6333823514171504, -2.532770108704866, -1.0917697438337792, 0.7542370444757319, 0.3814716845740165], [0.748155369385981, 1.4071864934871512, 0.17137172810409493, -0.1700089079408552, -0.39622893536494863, 0.16784139928529, -2.049211960993675, 1.5286220716692664, -2.112732667090163, -1.5375896628608756, 0.10520375129398785, 0.9682373202721709, 0.575650161655274, 1.5183576063815099, -0.4139949094942258, -0.6235065080529658]], "dtype": "float64"}, 4, false], "e": {"__tup__": [{"__nd__": [[0, 7, 1, 14], [12, 7, 0, 13], [15, 2, 6, 9], [5, 4, 12, 13], [9, 8, 6, 2], [6, 9, 8, 4], [0, 1, 7, 2], [6, 15, 2, 5], [10, 8, 4, 5], [12, 3, 4, 14], [1, 11, 13, 4], [2, 12, 15, 0], [7, 12, 3, 13], [9, 6, 4, 8], [12, 4, 1, 3], [12, 3, 4, 10], [7, 3, 12, 10], [7, 0, 14, 9], [7, 0, 3, 2], [9, 6, 8, 4], [14, 3, 12, 10], [0, 7, 13, 1], [6, 15, 8, 2], [5, 11, 9, 10], [12, 5, 2, 0], [6, 5, 9, 15], [7, 2, 1, 0], [2, 14, 9, 8], [8, 0, 6, 11], [7, 13, 11, 0], [13, 0, 2, 5], [9, 11, 1, 0], [3, 14, 8, 9], [7, 10, 13, 8], [6, 5, 9, 8], [10, 8, 5, 7], [11, 10, 5, 9], [4, 15, 10, 2], [5, 15, 11, 6], [9, 8, 2, 11], [9, 11, 6, 5], [7, 11, 1, 13], [8, 10, 6, 9], [12, 7, 13, 3], [0, 7, 13, 11], [5, 15, 9, 6], [12, 15, 5, 2], [1, 11, 13, 2], [3, 4, 6, 8], [9, 8, 10, 11], [8, 6, 9, 15], [8, 9, 6, 10], [2, 8, 14, 9], [4, 3, 1, 2], [9, 8, 14, 2], [14, 7, 0, 6], [12, 5, 15, 4], [15, 10, 4, 5], [12, 3, 1, 13], [9, 6, 8, 4], [5, 6, 13, 4], [13, 11, 5, 7], [12, 7, 3, 14], [5, 12, 10, 6]], "dtype": "int64"}, {"__nd__": [[0.429870255451149, 0.29689991635745244, 0.08913714815786143, 0.0869866240133998], [0.9402201636342562, 0.028401170130780168, 0.010265801075411501, 0.008708227097948036], [0.46847725525517464, 0.4398712751851556, 0.05380541306346501, 0.013111911195739802], [0.4676018803026605, 0.15851613773444093, 0.1332814742531436, 0.10018563339819475], [0.8093205855468738, 0.11222175455569644, 0.02889052648290729, 0.01512490475637244], [0.7230192788291872, 0.128043856695608, 0.051440590887341424, 0.025855787048660532], [0.3729595173593903, 0.19357066453049795, 0.19086698808327293, 0.08480289921966509], [0.5857115051907892, 0.14521707064297523, 0.08573632417499284, 0.08206152988758521], [0.489729577920213, 0.20122620432838387, 0.08328244164708556, 0.0743277451877279], [0.3678035626093572, 0.3517183554497145, 0.1571870059020465, 0.029080828614333892], [0.4988549383679222, 0.14774233229480388, 0.1380915482626468, 0.033443405163079254], [0.3897589847241321, 0.3620971526841612, 0.06562575417891565, 0.05064565738273941], [0.44214932605431545, 0.4047161585572366, 0.08726683786719219, 0.04118904734666359], [0.891719783013945, 0.06667414780155509, 0.02147634749921324, 0.01360022109465023], [0.3045522490649924, 0.23342114966843783, 0.10729245981327945, 0.0962009464733412], [0.8688413892595976, 0.08966680219842231, 0.03680724239848071, 0.002964399022340695], [0.6066803846471862, 0.15512555193242386, 0.1330543348580387, 0.03277132434203786], [0.22595926818348078, 0.14434495769715447, 0.11249952714463492, 0.09525678621368261], [0.6025354294945218, 0.18664181590418638, 0.06664511154297749, 0.031800602288881846], [0.3287550789792662, 0.308832178442815, 0.28563725071767454, 0.028495536265085105], [0.27427320493138246, 0.20906537254008678, 0.19087177475171904, 0.16056245848026215], [0.5298098524267181, 0.17965518920708795, 0.17447509894443056, 0.03985515769717989], [0.8150875322311215, 0.09133239921295903, 0.034216508684308734, 0.01808972738342683], [0.4387954694140403, 0.1647244450259063, 0.10468176862724705, 0.07416384069559927], [0.4757355963696873, 0.10697503792577419, 0.10667825433245268, 0.06851667480094412], [0.7586894297242698, 0.06250200740379254, 0.04798950286596846, 0.035480399810973846], [0.4191058968208051, 0.34988072297575734, 0.08002538216930807, 0.041909067043531964], [0.5150310793699245, 0.2648402799419306, 0.13572765393750152, 0.04414083033857892], [0.5520680708429875, 0.3012390545015296, 0.10167949517861877, 0.019685489094906906], [0.6037184458441897, 0.22053287252920115, 0.07909940049999946, 0.044064459167419585], [0.1727411775949032, 0.16282131573299619, 0.11698584361892105, 0.09323394007675855], [0.20134885395026725, 0.1498208329395667, 0.08029612038610617, 0.07816114574602552], [0.3588284396303297, 0.3070347981145519, 0.19307596013856476, 0.05660974368511458], [0.6754667118284761, 0.10864191729473446, 0.06276415483150224, 0.05207634683391826], [0.6009638453983304, 0.20505869264789686, 0.07608229562952583, 0.04742793654381366], [0.8119028219838801, 0.0765911070139417, 0.033427441626517874, 0.027137350058017057], [0.3014273114368967, 0.17242908553162334, 0.14712289093813855, 0.10130727313581145], [0.5215174762845629, 0.15561747155567004, 0.10425476133045336, 0.06389349707028451], [0.558716571152599, 0.17599873614672218, 0.0682904326628936, 0.05566314714251404], [0.8033963954753965, 0.09624362767494934, 0.031040705519780217, 0.021589719087233488], [0.5679492551917634, 0.21960903864286271, 0.11624816636040337, 0.03297435799493237], [0.49124643720558164, 0.14728088722320562, 0.1214961287723909, 0.10155741569133224], [0.26728491412333166, 0.2034211180949174, 0.1583452650335174, 0.1289549482929705], [0.5788454558973772, 0.16040852370255182, 0.11680232446401834, 0.051758054715956334], [0.374612089600426, 0.2868197615751016, 0.10714896785799369, 0.07539377837893864], [0.2163136038700121, 0.20614445592570796, 0.18684518292136365, 0.16289379473443258], [0.9081038657953195, 0.023281482817054604, 0.020562436221118273, 0.016825929794360824], [0.5318555614029625, 0.176515718728776, 0.09964478378459787, 0.09327320742370032], [0.6150747226601502, 0.15592686269789668, 0.09603515104657821, 0.03417966659875956], [0.40095202517579126, 0.18437154451657242, 0.12082026339743551, 0.11023843283792897], [0.6994109202580746, 0.1639583761176109, 0.0666530932985075, 0.04589724924895927], [0.6872825829838483, 0.1144919789052777, 0.10333906897828747, 0.0713320210293877], [0.24779889469715133, 0.1555559087043736, 0.1363965011539928, 0.13417021133012885], [0.2537890352231324, 0.21062816117408636, 0.1426706059372605, 0.09571287514164802], [0.4920846672212929, 0.16061215462755982, 0.13676876585961234, 0.13526414332946332], [0.21589166610843366, 0.13426238467240845, 0.10772249883935003, 0.0931797389164783], [0.3664248121556951, 0.20425804592593322, 0.17292445199122405, 0.12475508420085447], [0.5151736483958761, 0.26151841085816624, 0.06421375240299952, 0.04543720363657975], [0.9304239878975676, 0.013187276309929025, 0.012632878526917458, 0.010772371508507262], [0.7678136854470473, 0.11787215118559478, 0.107768812686685, 0.0023392005377093315], [0.5421088655966907, 0.14705324940067116, 0.10774128921708186, 0.08868981676145772], [0.6134402791953765, 0.11458692110210617, 0.10610539524514985, 0.10150482350812047], [0.4680997551056247, 0.2961924960468912, 0.21686205189848476, 0.004558843307939385], [0.8653196237933765, 0.06803267146565807, 0.021876178334557194, 0.020679539199918075]], "dtype": "float64"}, {"__nd__": [15, 11, 18, 13, 16, 19, 21, 18, 21, 23, 14, 14, 16, 15, 10, 12], "dtype": "int64"}, 1.0717085346969617]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(route, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")